<font color="#1aac39" size='6px'><b>Video Smart Redaction (Temporal Privacy Redaction)</b></font>

<font color="#1aac39" size='4px'><b>The Pipeline Architecture</b></font>

The system operates in a temporal privacy-focused pipeline designed to track and redact sensitive information across video streams with pixel-level precision. It leverages **SAM 3 Video Predictor** to ensure that once a target is identified, it remains obscured throughout the entire sequence, even during complex motion .

<font color="#D35400" size='4px'><b>Stage I: Targeted Security Discovery (Frame 0)</b></font>
Before tracking begins, the system identifies privacy targets on the first frame of the video.

**Automatic Security Protocol:** By default, it iterates through the `SMART_REDACTION_OBJECTS` list, which includes:
* <font color="#884EA0"><i>Identity:</i></font> "face", "signature", "id card", "document".
* <font color="#884EA0"><i>Digital/Vehicular:</i></font> "license plate", "laptop screen", "phone screen", "credit card".

**Custom Redaction:** Users can specify a singular target (e.g., `prompt_text="license plate"`) to focus the tracking engine strictly on that specific class.

<font color="#D35400" size='4px'><b>Stage II: SAM 3 Video Tracker Initialization</b></font>
The system initializes a tracking session to maintain temporal consistency.

**Session Management:** A unique `session_id` is created using `models.load_sam3_video_model` .

**Spatial Prompting:** The target nouns are sent as text prompts to Frame 0. SAM 3 generates the initial high-resolution masks for every instance detected at the start of the video.

<font color="#D35400" size='4px'><b>Stage III: Temporal Propagation (Privacy Persistence)</b></font>
To avoid re-detecting (which causes "flickering" redaction), the system uses **Mask Propagation**.

**Memory Bank:** SAM 3 builds a visual memory of the sensitive object. As the video progresses, the model "propels" the mask forward into new frames.

**Movement Adaptation:** The system automatically adjusts for changes in scale, rotation, and lighting, ensuring the redaction mask sticks to the target even if it moves or turns away.

**Occlusion Handling:** If a sensitive object is temporarily hidden (e.g., a car driving behind a tree), the memory bank helps the tracker re-identify and immediately re-blur the object when it reappears.

<font color="#D35400" size='4px'><b>Stage IV: Aggregation & Mask Refinement</b></font>
Video tracking results are aggregated frame-by-frame.

**Confidence Filtering:** Only masks exceeding the `SAM3_CONF_SMART_REDACTION_VIDEO_THRESHOLD` are kept to prevent blurring background noise.

**Multi-Class Merging:** If the system is tracking multiple privacy targets (e.g., both faces and license plates), it performs a **Boolean OR** operation on all masks for each frame to create a comprehensive "Redaction Map" for that specific timestamp.

<font color="#D35400" size='4px'><b>Stage V: Temporal Gaussian Redaction & Rendering</b></font>
The final secure video is rendered by applying the redaction map to the original stream.

**Selective Blurring:** The system applies a heavy `cv2.GaussianBlur` (typically kernel 51+) only to the pixels identified in the Redaction Map .

**Video Assembly:** The redacted frames are stitched back together using `cv2.VideoWriter`, resulting in a privacy-compliant video where sensitive data is structurally unrecoverable, while the rest of the scene remains 100% visible and sharp.

<font color="#2E86C1" size='6px'>Environment Initialization & Repository Setup</font>

In [ ]:
COLAB = False

In [ ]:
import os
import sys

if COLAB:
    # Clone Repository 
    if not os.path.exists("sam3-toolkit"):
        print("--- Cloning repository...")
        !git clone https://github.com/MrKGhasemi/sam3-toolkit.git
    else:
        print("--- Repository already exists.")

    # Auto-Detect Paths 
    repo_root = os.path.abspath("sam3-toolkit")
    sam3_install_path = None
    practical_path = None

    for root, dirs, files in os.walk(repo_root):
        if ("sam3" in dirs or "sam3_main" in root or root == repo_root):
            if sam3_install_path is None or len(root) < len(sam3_install_path):
                sam3_install_path = root

        if "smart_redaction.py" in files:
            practical_path = root

    if not sam3_install_path or not practical_path:
        raise FileNotFoundError(
            "     --- Critical paths (sam3 or practical) not found.")

    print(f"Found SAM 3 Library at: {sam3_install_path}")
    print(f"Found Project Code at: {practical_path}")

    # Install Dependencies 
    print("--- Installing Python packages...")
    !{sys.executable} -m pip install -q opencv-python matplotlib scikit-learn transformers spacy openai gdown mega.py huggingface_hub
    !{sys.executable} -m pip install -q einops decord pycocotools

    # Install SAM 3 
    print("--- Installing SAM 3 Library...")
    !{sys.executable} -m pip install -e "{sam3_install_path}"

    # Configure Working Directory 
    os.chdir(practical_path)
    if practical_path not in sys.path:
        sys.path.insert(0, practical_path)
    print(f"--- Working directory set to: {os.getcwd()}")

    # Download Model Weights
    weights_dir = "weights"
    weights_path = os.path.join(weights_dir, "sam3.pt")
    os.makedirs(weights_dir, exist_ok=True)

    # If the model is Private/Gated, you MUST provide a token.
    # Get it here: https://huggingface.co/settings/tokens
    HF_REPO_ID = "facebook/sam3" 
    HF_FILENAME = "sam3.pt"               
    HF_TOKEN = "HF_TOKEN"  # Replace with your actual token or set to None

    if not os.path.exists(weights_path):
        print(f"--- Starting download...")

        try:
            from huggingface_hub import hf_hub_download
            print("--- Connecting to Hugging Face Hub...")
            downloaded_file = hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=HF_FILENAME,
                local_dir=weights_dir,
                token=HF_TOKEN,
                local_dir_use_symlinks=False  # Ensure we get the actual file, not a symlink
            )
            if os.path.basename(downloaded_file) != "sam3.pt":
                os.rename(downloaded_file, weights_path)

            print("--- Download attempt finished.")

        except Exception as e:
            print(f"      --- Error downloading: {e}")
            print("           Tip: If using Hugging Face private repo, ensure HF_TOKEN is set.")

    else:
        print("--- Weights file already exists.")

    # Verify File Integrity 
    if os.path.exists(weights_path):
        final_size = os.path.getsize(weights_path) / (1024 * 1024)
        print(f"--- Final File Size: {final_size:.2f} MB")
        if final_size < 2000:
            print(
                "      --- WARNING: File seems too small (<2GB). It might be corrupt or a placeholder.")
    else:
        print("      --- Error: File not found after download.")

    # Download SpaCy Model 
    print("--- Checking SpaCy model...")
    !{sys.executable} -m spacy download en_core_web_lg

    # Verify Import 
    print("\n--- Verifying imports...")
    repo_root = os.path.abspath("/sam3-toolkit")

    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
        print(f"--- Added '{repo_root}' to sys.path")

    practical_dir = os.path.join(repo_root, "practical")
    if practical_dir not in sys.path:
        sys.path.insert(0, practical_dir)
        print(f"--- Added '{practical_dir}' to sys.path")

    # Verify all dependencies
    print("\n--- Retrying imports...")
    try:
        import sam3
        print("--- 'sam3' library loaded!")
    except ImportError as e:
        print(f"      --- Failed to load sam3: {e}")
        print("       --- Attempting hard install fix...")
        !pip install "{repo_root}"
        import sam3

    try:
        import models
        print("--- 'models.py' loaded!")
    except ImportError as e:
        print(f"      --- Failed to load models: {e}")

    !pip uninstall numpy -y
    !{sys.executable} -m pip install "numpy<2.0"

    # Check GPU
    import torch
    print(f"--- GPU Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"--- GPU Name: {torch.cuda.get_device_name(0)}")

    if os.path.exists(practical_dir):
        os.chdir(practical_dir)
        print(f"--- Current Directory: {os.getcwd()}")


---

<font color="#117A65" size='6px'>Loading Core Modules & Dependencies</font>

<font color="#D68910" size='4px'>*smart_redaction:*</font> Contains the main pipeline for generate privacy masks.

<font color="#D68910" size='4px'>*visualization:*</font> Handles the drawing of bounding boxes, masks, and legends.



In [ ]:
import os
import sys
root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root)

if os.path.basename(os.getcwd()) == "ipynb":
    os.chdir(root)

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
import smart_redaction
import torch
import torchvision
import utils

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())

In [ ]:
import numpy
print(numpy.__version__) 
# process is Not capable with numpy 2.0.0 or higher

```Python
# From configs.py 
LLM_CAPTION_MODEL_NAME = "gpt-5-chat"
SAM3_CONF_SMART_REDACTION_VIDEO_THRESHOLD = 0.2
```

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class MockArgs:
    def __init__(self, mode, output_dir, api_key=None, base_url=None):
        self.mode = mode
        self.output = output_dir
        self.api_key = api_key
        self.base_url = base_url


output_directory = "notebook_outputs"
os.makedirs(output_directory, exist_ok=True)

args = MockArgs(mode="blip", output_dir=output_directory)

# for LLM mode:
# args = MockArgs(
#     mode="llm",
#     output_dir=output_directory,
#     api_key="api_key",
#     base_url="base_url"
# )

---

<font color="#1aac39" size='6px'>Video Execution: Privacy Redaction Tracking</font>

This cell executes the Temporal Privacy Pipeline. By calling <font color="#1aac39" size='4px'>smart_redaction.smart_redaction_video</font>, the system leverages the SAM 3 Video Predictor to maintain a persistent "privacy shield" over sensitive objects across every frame.

The function orchestrates a multi-pass tracking logic:

<font color="#1aac39" size='4px'><i>Multi-Class Scoping:</i></font> The system iterates through the <font color="#1aac39"><i>SMART_REDACTION_OBJECTS</i></font> list (faces, license plates, etc.). For each category, it starts a dedicated SAM 3 session to ensure that even if different sensitive objects overlap, they are all correctly identified.

<font color="#1aac39" size='4px'><i>Temporal Propagation:</i></font> Instead of running a heavy detector on every frame, the system uses a Propagation Stream. It identifies the targets on Frame 0 and then "propels" those masks forward in time, utilizing SAM 3's internal memory to stay locked onto the objects despite motion or camera shakes.

<font color="#1aac39" size='4px'><i>Confidence Filtering:</i></font> To avoid blurring background noise, the system validates the tracking quality on every frame. It only adds a mask to the <font color="#1aac39"><i>cumulative_frame_masks</i></font> registry if the object score exceeds the <font color="#1aac39"><i>SAM3_CONF_SMART_REDACTION_VIDEO_THRESHOLD</i></font>.


```python
# Video Propagation Logic from smart_redaction.py
with torch.inference_mode():
    stream = predictor.handle_stream_request(request=dict(
        type="propagate_in_video", session_id=session_id
    ))

    for r in tqdm(stream, total=total_frames, desc=f"Tracking {noun}"):
        f_idx = r["frame_index"]
        out_data = r["outputs"]
        
        # Extract binary masks and scores
        masks_val = out_data.get("out_binary_masks")
        scores_val = out_data.get("out_obj_scores")
        
        # Logic to aggregate valid masks into cumulative_frame_masks[f_idx]
```        

In [ ]:
VIDEO_PATH = "videos/crowd.mp4" 

<font color="#CB4335" size='6px'>Privacy Rendering & Temporal Blurring</font>

The final stage of the pipeline transforms the raw tracking data into a secure video output. The system applies the visual redaction layer by compositing the original stream with blurred variants.

The rendering process involves three key layers:

<font color="#CB4335" size='4px'><i>Bitwise Mask Aggregation:</i></font> For every frame, the system performs a Logical OR operation on all masks collected across different tracking passes. This creates a unified "redaction map" representing every pixel that must be obscured for that specific timestamp.

<font color="#CB4335" size='4px'><i>Gaussian Compositing:</i></font> The system creates a heavily blurred version of the current frame using <font color="#CB4335"><i>cv2.GaussianBlur</i></font>. It then performs a high-speed replacement: only the pixels inside the aggregated mask are replaced by the blurred ones, leaving the non-sensitive background perfectly sharp.

<font color="#CB4335" size='4px'><i>Stream Writing:</i></font> The processed frames are fed into a <font color="#CB4335"><i>cv2.VideoWriter</i></font> using the mp4v codec. This ensures that the privacy protection is baked into the video file itself, making the redacted information structurally unrecoverable.

```python
# Core Redaction Logic from smart_redaction.py
def apply_redaction(image, masks_list, kernel_size=(51, 51)):
    output_image = image.copy()
    combined_mask = np.zeros(image.shape[:2], dtype=bool)

    # Aggregate all detected objects into one mask
    for m in masks_list:
        combined_mask = np.logical_or(combined_mask, m['segmentation'])

    # Apply heavy blur only to the sensitive areas
    blurred_image = cv2.GaussianBlur(image, kernel_size, 0)
    output_image[combined_mask] = blurred_image[combined_mask]
    
    return output_image
```    

In [ ]:
output_information = smart_redaction.smart_redaction_video(VIDEO_PATH, args, prompt_text="face", blur_strength=90)

In [ ]:
output_information = smart_redaction.smart_redaction_video(VIDEO_PATH, args, prompt_text=None, blur_strength=90)

<font color="#2874A6" size='6px'>Universal Data Serialization</font>

This cell executes the Data Compression & Archiving step. Regardless of the specific task (Counting, Segmentation, or Redaction), the results—often containing thousands of high-resolution binary masks—are massive. This function efficiently packs them for storage.

The function <font color="#2874A6"><i>utils.compress_and_save_output_information</i></font> applies a rigorous optimization pipeline:

<font color="#2874A6" size='4px'><i>RLE Compression:</i></font> Instead of saving dense boolean arrays (which consume huge disk space), the function converts every segmentation mask into <font color="#2874A6"><i>Run-Length Encoding (RLE)</i></font> using <font color="#2874A6"><i>pycocotools</i></font>. This compresses the data by storing counts of consecutive pixels rather than the pixels themselves.

<font color="#2874A6" size='4px'><i>Memory Optimization:</i></font> Before encoding, the binary masks are converted to <font color="#2874A6"><i>Fortran-contiguous arrays</i></font> via <font color="#2874A6"><i>np.asfortranarray</i></font>. This specific memory layout is required for high-speed C-based encoding operations, ensuring the compression process doesn't bottleneck the pipeline.

<font color="#2874A6" size='4px'><i>Structured Pickling:</i></font> The simplified, compressed dictionary is serialized using Python's <font color="#2874A6"><i>pickle</i></font> module. This preserves the complex nested structure (frames -> objects -> metadata) in a single file that can be instantly reloaded for analysis later.

In [ ]:
save_path = os.path.join(output_directory, "smart_redaction_output_crowd_compressed.pkl")

utils.compress_and_save_output_information(output_information, save_path)